<a href="https://colab.research.google.com/github/naman-0804/learning/blob/Langchain_Langraph/data_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!pip install -U langchain-google-genai langchain-chroma chromadb
!pip install langchain_community
!pip install pypdf
#!pip instal bs4
!pip install langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 7.4 MB/s eta 0:00:00
^C


In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("attention(1).pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=10
)
text = text_splitter.split_documents(documents)

In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=text,
    embedding=embeddings,
    collection_name="attention_docs"
)

In [13]:
results = vectorstore.similarity_search(
    "what is the main formula of topic?",
    k=1
)

for result in results:
    print(result.page_content)

Encoder hidden states → calculate attention scores → convert scores to weights → weighted sum →
context vector → decoder output
4. The Main Equations
Step 1 — Score
score(q, k■) = similarity between the query q and key k■
Step 2 — Softmax
α■ = exp(score■) / Σ■ exp(score■)
Step 3 — Weighted sum
context = Σ■ α■ v■
Here, Q = Query represents what the model is currently looking for, K = Key represents what each token


In [10]:
collection = vectorstore._collection
collection

Collection(name=attention_docs)

In [11]:
data = collection.get()

print(data["documents"])

['Attention Mechanism\nA practical note for NLP, RNNs, LSTMs and Transformers\n1. Why Attention Was Needed\nIn a basic encoder-decoder RNN/LSTM, the encoder processes the entire input sequence and the\ndecoder traditionally relies heavily on a single final context vector. For long sentences, compressing all\ninformation into one vector can lose important details.\nAttention solves this by allowing the decoder to look at different encoder hidden states and assign\ndifferent importance to them.', '2. Core Idea\nInstead of treating every input word equally, attention calculates a score for each relevant encoder state.\nHigher score = more attention. The scores are converted into weights, and a weighted sum produces the\ncontext vector.\nExample: translating "I love machine learning". When generating the translated word corresponding to\n"machine", the model can give a high attention weight to the encoder state for "machine" and lower\nweights to unrelated words.\n3. Basic Attention Flow',

In [14]:
data = collection.get(include=["embeddings"])

print(data["embeddings"][0])

[ 0.00350655  0.00484968  0.00227572 ... -0.01244838 -0.0040459
 -0.00318704]


In [15]:
data = collection.get(
    include=["documents", "metadatas", "embeddings"]
)

print("Number of chunks:", len(data["documents"]))

print("\nTEXT:")
print(data["documents"][0])

print("\nMETADATA:")
print(data["metadatas"][0])

print("\nEMBEDDING:")
print(data["embeddings"][0])

Number of chunks: 9

TEXT:
Attention Mechanism
A practical note for NLP, RNNs, LSTMs and Transformers
1. Why Attention Was Needed
In a basic encoder-decoder RNN/LSTM, the encoder processes the entire input sequence and the
decoder traditionally relies heavily on a single final context vector. For long sentences, compressing all
information into one vector can lose important details.
Attention solves this by allowing the decoder to look at different encoder hidden states and assign
different importance to them.

METADATA:
{'page': 0, 'title': '(anonymous)', 'producer': 'ReportLab PDF Library - (opensource)', 'subject': '(unspecified)', 'author': '(anonymous)', 'keywords': '', 'page_label': '1', 'trapped': '/False', 'total_pages': 2, 'moddate': '2026-08-13T08:29:52+00:00', 'creationdate': '2026-08-13T08:29:52+00:00', 'creator': '(unspecified)', 'source': 'attention(1).pdf'}

EMBEDDING:
[ 0.00350655  0.00484968  0.00227572 ... -0.01244838 -0.0040459
 -0.00318704]
